# EEI6366 Lab 01: Big Data Processing with PySpark

**Dataset:** Open University Student Performance (`open_university_student_performance.csv`)  
**Size:** 150,500 rows × 9 columns  
**Source:** Generated / provided for EEI6366 (Open University of Sri Lanka student performance across centers and courses)

This notebook fulfills **Step 2 – Prepare and Do the Lab**:
1. Study Getting Started with PySpark (completed)
2. Use a dataset with ≥ 100,000 rows
3. Explore with **PySpark DataFrame API** (≥ 3 operations)
4. Explore with **Spark SQL** (≥ 3 operations)

Columns: `student_id`, `course_code`, `academic_year`, `center`, `continuous_assessment_mark`, `final_exam_mark`, `overall_mark`, `grade`, `status`

## 1. Install & Initialize PySpark (Google Colab style)

In [1]:
# Run this cell only if you are on Google Colab and PySpark is not already available
!pip install -q pyspark findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (SparkSession
         .builder
         .appName("EEI6366-Lab01-Student-Performance")
         .master("local[*]")
         .getOrCreate())

spark

## 2. Load the Dataset

In [3]:
from google.colab import files
from pyspark.sql import SparkSession

# Step 1: Upload CSV file
print("Please upload your CSV dataset:")
uploaded = files.upload()

# Step 2: Get the uploaded filename automatically
file_name = list(uploaded.keys())[0]

# Step 3: Read the uploaded CSV using PySpark
df = spark.read.csv(
    file_name,
    header=True,
    inferSchema=True
)

# Step 4: Display dataset information
print("\nDataset:", file_name)
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

# Step 5: Display schema
df.printSchema()

# Step 6: Display first 5 rows
df.show(5, truncate=False)

Please upload your CSV dataset:


Saving open_university_student_performance.csv to open_university_student_performance.csv

Dataset: open_university_student_performance.csv
Total rows: 150500
Total columns: 9
root
 |-- student_id: string (nullable = true)
 |-- course_code: string (nullable = true)
 |-- academic_year: string (nullable = true)
 |-- center: string (nullable = true)
 |-- continuous_assessment_mark: double (nullable = true)
 |-- final_exam_mark: double (nullable = true)
 |-- overall_mark: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- status: string (nullable = true)

+----------+-----------------------------+-------------+----------+--------------------------+---------------+------------+-----+------+
|student_id|course_code                  |academic_year|center    |continuous_assessment_mark|final_exam_mark|overall_mark|grade|status|
+----------+-----------------------------+-------------+----------+--------------------------+---------------+------------+-----+------+
|S720061358|EEX

## 3. PySpark DataFrame API Operations (≥ 3)

### Operation 1 – Basic exploration & descriptive statistics

In [4]:
# Schema already printed above
# Summary statistics for numeric columns
df.select(
    "continuous_assessment_mark",
    "final_exam_mark",
    "overall_mark"
).describe().show()

# Distinct values for categorical columns
print("Unique courses:", df.select("course_code").distinct().count())
print("Unique centers:", df.select("center").distinct().count())
print("Academic years:")
df.select("academic_year").distinct().orderBy("academic_year").show()

+-------+--------------------------+-----------------+------------------+
|summary|continuous_assessment_mark|  final_exam_mark|      overall_mark|
+-------+--------------------------+-----------------+------------------+
|  count|                    150500|           150500|            150500|
|   mean|         64.95519069767434|57.86662392026619|60.702051827242855|
| stddev|        14.818542161387795|17.60697293991903|12.107955949253252|
|    min|                      15.0|             10.0|              13.0|
|    max|                      98.0|             95.0|              96.2|
+-------+--------------------------+-----------------+------------------+

Unique courses: 6
Unique centers: 6
Academic years:
+-------------+
|academic_year|
+-------------+
|    2023/2024|
|    2024/2025|
|    2025/2026|
+-------------+



### Operation 2 – Filtering + Ordering

In [5]:
# Students who failed the EEI6366 - Big Data Processing course, ordered by overall_mark ascending
failed_bigdata = (df
    .filter(
        (F.col("course_code") == "EEI6366 - Big Data Processing") &
        (F.col("status") == "Fail")
    )
    .orderBy(F.col("overall_mark").asc())
    .select("student_id", "center", "academic_year",
            "continuous_assessment_mark", "final_exam_mark",
            "overall_mark", "grade")
)

print("Number of Failures in EEI6366:", failed_bigdata.count())
failed_bigdata.show(10, truncate=False)

# High performers (overall_mark >= 80) from Colombo center
high_colombo = df.filter(
    (F.col("center") == "Colombo") &
    (F.col("overall_mark") >= 80)
).orderBy(F.desc("overall_mark"))

print("\nHigh performers in Colombo (>=80):", high_colombo.count())
high_colombo.show(5)

Number of Failures in EEI6366: 2492
+----------+----------+-------------+--------------------------+---------------+------------+-----+
|student_id|center    |academic_year|continuous_assessment_mark|final_exam_mark|overall_mark|grade|
+----------+----------+-------------+--------------------------+---------------+------------+-----+
|S732998941|Kandy     |2025/2026    |31.9                      |10.0           |18.8        |F    |
|S743192578|Batticaloa|2023/2024    |34.8                      |10.0           |19.9        |F    |
|S730758614|Colombo   |2025/2026    |32.8                      |13.4           |21.2        |F    |
|S767933733|Colombo   |2023/2024    |37.2                      |11.6           |21.8        |F    |
|S781968844|Batticaloa|2025/2026    |18.1                      |25.1           |22.3        |F    |
|S728906265|Colombo   |2025/2026    |41.3                      |10.0           |22.5        |F    |
|S706904305|Kurunegala|2023/2024    |21.6                      |

### Operation 3 – Aggregations (GroupBy + Agg)

In [6]:
# Average marks and pass rate by Center
center_stats = (df
    .groupBy("center")
    .agg(
        F.count("*").alias("total_students"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall_mark"),
        F.round(F.avg("continuous_assessment_mark"), 2).alias("avg_ca"),
        F.round(F.avg("final_exam_mark"), 2).alias("avg_exam"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall_mark"))
)

print("Performance by Regional Center:")
center_stats.show(truncate=False)

# Average marks and pass rate by Course
course_stats = (df
    .groupBy("course_code")
    .agg(
        F.count("*").alias("enrollments"),
        F.round(F.avg("overall_mark"), 2).alias("avg_overall"),
        F.round(
            F.sum(F.when(F.col("status") == "Pass", 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("pass_rate_pct")
    )
    .orderBy(F.desc("avg_overall"))
)

print("\nPerformance by Course:")
course_stats.show(truncate=False)

Performance by Regional Center:
+----------+--------------+----------------+------+--------+-------------+
|center    |total_students|avg_overall_mark|avg_ca|avg_exam|pass_rate_pct|
+----------+--------------+----------------+------+--------+-------------+
|Colombo   |25125         |60.78           |65.06 |57.92   |90.06        |
|Jaffna    |24963         |60.75           |64.92 |57.97   |90.17        |
|Matara    |25127         |60.71           |64.94 |57.9    |90.15        |
|Batticaloa|24941         |60.71           |65.07 |57.81   |89.9         |
|Kandy     |25188         |60.68           |64.89 |57.87   |89.88        |
|Kurunegala|25156         |60.58           |64.84 |57.74   |89.67        |
+----------+--------------+----------------+------+--------+-------------+


Performance by Course:
+------------------------------+-----------+-----------+-------------+
|course_code                   |enrollments|avg_overall|pass_rate_pct|
+------------------------------+-----------+-------

### Bonus Operation – Derived column + Window function (ranking)

In [7]:
# Create a new column: performance gap (CA - Exam)
df_with_gap = df.withColumn(
    "ca_exam_gap",
    F.round(F.col("continuous_assessment_mark") - F.col("final_exam_mark"), 1)
)

# Rank students within each course by overall_mark (top performers)
window_spec = Window.partitionBy("course_code").orderBy(F.desc("overall_mark"))

ranked = (df_with_gap
    .withColumn("rank_in_course", F.rank().over(window_spec))
    .filter(F.col("rank_in_course") <= 3)   # Top 3 per course
    .select("course_code", "student_id", "center", "overall_mark",
            "grade", "ca_exam_gap", "rank_in_course")
    .orderBy("course_code", "rank_in_course")
)

print("Top 3 students per course:")
ranked.show(20, truncate=False)

Top 3 students per course:
+-----------------------------+----------+----------+------------+-----+-----------+--------------+
|course_code                  |student_id|center    |overall_mark|grade|ca_exam_gap|rank_in_course|
+-----------------------------+----------+----------+------------+-----+-----------+--------------+
|EEI6366 - Big Data Processing|S724420136|Batticaloa|96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S737868508|Kandy     |96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S758463129|Batticaloa|96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S715827064|Kurunegala|96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S778755345|Colombo   |96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S735312963|Kurunegala|96.2        |A+   |3.0        |1             |
|EEI6366 - Big Data Processing|S767976517|Jaffna    |96.2        |A+   |3

## 4. Spark SQL Operations (≥ 3)

First register the DataFrame as a temporary view.

In [8]:
df.createOrReplaceTempView("student_performance")

# Verify
spark.sql("SELECT COUNT(*) AS total_rows FROM student_performance").show()

+----------+
|total_rows|
+----------+
|    150500|
+----------+



### SQL Operation 1 – Filtering + Aggregation

In [9]:
# Pass rate and average overall mark by Academic Year and Center
spark.sql("""
SELECT
    academic_year,
    center,
    COUNT(*) AS students,
    ROUND(AVG(overall_mark), 2) AS avg_overall,
    ROUND(SUM(CASE WHEN status = 'Pass' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pass_rate_pct
FROM student_performance
GROUP BY academic_year, center
ORDER BY academic_year, avg_overall DESC
""").show(30, truncate=False)

+-------------+----------+--------+-----------+-------------+
|academic_year|center    |students|avg_overall|pass_rate_pct|
+-------------+----------+--------+-----------+-------------+
|2023/2024    |Batticaloa|8417    |60.9       |89.72        |
|2023/2024    |Jaffna    |8259    |60.85      |90.20        |
|2023/2024    |Matara    |8485    |60.8       |90.24        |
|2023/2024    |Colombo   |8379    |60.78      |90.01        |
|2023/2024    |Kandy     |8460    |60.73      |90.25        |
|2023/2024    |Kurunegala|8367    |60.55      |89.65        |
|2024/2025    |Colombo   |8431    |60.77      |89.70        |
|2024/2025    |Kandy     |8265    |60.7       |89.91        |
|2024/2025    |Jaffna    |8255    |60.67      |90.21        |
|2024/2025    |Matara    |8281    |60.62      |89.88        |
|2024/2025    |Kurunegala|8376    |60.59      |89.45        |
|2024/2025    |Batticaloa|8231    |60.59      |90.30        |
|2025/2026    |Colombo   |8315    |60.78      |90.46        |
|2025/20

### SQL Operation 2 – Conditional logic (CASE) + Grade distribution

In [10]:
# Grade distribution for the course EEI6366 - Big Data Processing
spark.sql("""
SELECT
    grade,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
GROUP BY grade
ORDER BY
    CASE grade
        WHEN 'A+' THEN 1 WHEN 'A' THEN 2 WHEN 'A-' THEN 3
        WHEN 'B+' THEN 4 WHEN 'B' THEN 5 WHEN 'B-' THEN 6
        WHEN 'C+' THEN 7 WHEN 'C' THEN 8 WHEN 'C-' THEN 9
        WHEN 'D+' THEN 10 WHEN 'D' THEN 11 WHEN 'F' THEN 12
        ELSE 13
    END
""").show()

+-----+-----+----------+
|grade|count|percentage|
+-----+-----+----------+
|   A+|  527|      2.09|
|    A| 2596|     10.31|
|   A-| 2656|     10.55|
|   B+| 3531|     14.03|
|    B| 4010|     15.93|
|   B-| 3929|     15.61|
|   C+| 3166|     12.58|
|    C| 2261|      8.98|
|   C-| 1363|      5.42|
|   D+|  702|      2.79|
|    D|  288|      1.14|
|    F|  139|      0.55|
+-----+-----+----------+



### SQL Operation 3 – Window function (moving / ranking style)

In [11]:
# For each student in a center, show their overall_mark and the average overall_mark of that center
# plus the difference from the center average (using window)
spark.sql("""
SELECT
    student_id,
    center,
    course_code,
    overall_mark,
    ROUND(AVG(overall_mark) OVER (PARTITION BY center), 2) AS center_avg,
    ROUND(overall_mark - AVG(overall_mark) OVER (PARTITION BY center), 2) AS diff_from_center_avg
FROM student_performance
WHERE course_code = 'EEI6366 - Big Data Processing'
ORDER BY center, overall_mark DESC
LIMIT 20
""").show(truncate=False)

+----------+----------+-----------------------------+------------+----------+--------------------+
|student_id|center    |course_code                  |overall_mark|center_avg|diff_from_center_avg|
+----------+----------+-----------------------------+------------+----------+--------------------+
|S724420136|Batticaloa|EEI6366 - Big Data Processing|96.2        |60.78     |35.42               |
|S758463129|Batticaloa|EEI6366 - Big Data Processing|96.2        |60.78     |35.42               |
|S794030723|Batticaloa|EEI6366 - Big Data Processing|96.2        |60.78     |35.42               |
|S778153253|Batticaloa|EEI6366 - Big Data Processing|94.2        |60.78     |33.42               |
|S783908332|Batticaloa|EEI6366 - Big Data Processing|93.1        |60.78     |32.32               |
|S714375621|Batticaloa|EEI6366 - Big Data Processing|92.4        |60.78     |31.62               |
|S746876404|Batticaloa|EEI6366 - Big Data Processing|91.8        |60.78     |31.02               |
|S71263568

### SQL Operation 4 (extra) – Finding interesting patterns

In [12]:
# Students whose Continuous Assessment is strong (>=70) but Final Exam is weak (<40)
# – possible exam anxiety or preparation issues
spark.sql("""
SELECT
    student_id,
    course_code,
    center,
    continuous_assessment_mark AS ca,
    final_exam_mark AS exam,
    overall_mark,
    grade,
    status
FROM student_performance
WHERE continuous_assessment_mark >= 70
  AND final_exam_mark < 40
ORDER BY continuous_assessment_mark DESC
LIMIT 15
""").show(truncate=False)

+----------+------------------------------+----------+----+----+------------+-----+------+
|student_id|course_code                   |center    |ca  |exam|overall_mark|grade|status|
+----------+------------------------------+----------+----+----+------------+-----+------+
|S778749962|EEI6366 - Big Data Processing |Jaffna    |98.0|37.7|61.8        |B    |Pass  |
|S795707625|EEI6366 - Big Data Processing |Kurunegala|98.0|29.4|56.8        |B-   |Pass  |
|S717909408|EEX6340 - Machine Learning    |Jaffna    |98.0|30.2|57.3        |B-   |Pass  |
|S736556540|EEI6366 - Big Data Processing |Kurunegala|98.0|31.6|58.2        |B-   |Pass  |
|S794807380|EEX5565 - Software Engineering|Kandy     |98.0|37.3|61.6        |B    |Pass  |
|S760714213|EEX6340 - Machine Learning    |Colombo   |98.0|16.2|48.9        |C    |Pass  |
|S777253507|EEX5565 - Software Engineering|Batticaloa|98.0|14.8|48.1        |C    |Pass  |
|S722233828|EEX3373 - Web Development     |Colombo   |98.0|25.4|54.4        |C+   |Pass  |

## 5. Key Findings (for Padlet / Presentation)

- Dataset contains **150,500** student-course records across **6 courses**, **6 regional centers** and **3 academic years**.
- Overall pass rate is high (~90%).
- Performance varies by center; some centers show higher average overall marks and pass rates.
- There is a noticeable group of students with strong Continuous Assessment but weak Final Exam performance – useful insight for teaching support.
- Window functions and group-by aggregations make it easy to rank students and compute center/course level KPIs at scale.

## 6. Challenges

- Setting up the Spark environment correctly (especially Java & SPARK_HOME on local machines).
- Inferring schema correctly and handling any missing values (this dataset is clean).
- Choosing meaningful aggregations that produce actionable insights rather than just running random queries.

---
**Repository:** https://github.com/missarii/Big-Data-Processing-with-PySpark  
End of Lab 01 exploration.

In [13]:
# Stop the Spark session when finished
spark.stop()
print("Spark session stopped.")

Spark session stopped.
